# Zameen.com Real Estate Web Scraper

This notebook scrapes residential property listings from Zameen.com
for Faisalabad, Lahore, and Islamabad.

The first 10 pages of each city are scraped.

### Extracted fields
- Price
- Area
- Bathrooms
- Bedrooms
- Location

In [2]:
# import all libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

### Fetch links

In [3]:
# Define a function to scrape the links of cities

# We have to scrape the links of First 10 pages

def get_all_links(url):
  # list to store the links
  all_links=[]
  for i in range(1,11):
    page_url = url.format(i)
    response = requests.get(page_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    # extract the lists of all li on the page
    li= soup.find_all('li',class_='a37d52f0')
    base_url='https://www.zameen.com'
    for links in li:
      link=(links.find('a')['href'])
      # combine the url
      new_url=urljoin(base_url,link)
      # store all links in the list
      all_links.append(new_url)
  return all_links

In [4]:
# call the function to get all the links

# Faisalabad
fsd_url = 'https://www.zameen.com/Houses_Property/Faisalabad-16-{}.html'
fsd_links=get_all_links(fsd_url)

# Lahore
lhr_url = 'https://www.zameen.com/Houses_Property/Lahore-1-{}.html'
lhr_links=get_all_links(lhr_url)

# Islamabad
isd_url='https://www.zameen.com/Houses_Property/Islamabad-3-{}.html'
isd_links=get_all_links(isd_url)

### Fetch data

In [5]:
# create a function to fetch the data
def get_data(links):
  # create separate lists
  price_list=[]
  bath_list=[]
  area_list=[]
  bedroom_list=[]
  location_list=[]

# Call each link in the list of links
  for link in links:
    response = requests.get(link)
    soup = BeautifulSoup(response.text, 'html.parser')
    all_spans=soup.find_all('span',class_='_2fdf7fc5')
    price_list.append(all_spans[1].text)
    bath_list.append(all_spans[2].text)
    area_list.append(all_spans[3].text)
    bedroom_list.append(all_spans[5].text)
    location_list.append(all_spans[7].text)

  # return all lists
  return  price_list,bath_list,area_list,bedroom_list,location_list


In [6]:
# data for Faisalabad
price_list1,bath_list1,area_list1,bedroom_list1,location_list1=get_data(fsd_links)

In [7]:
# data for Lahore
price_list2,bath_list2,area_list2,bedroom_list2,location_list2=get_data(lhr_links)

In [8]:
# data for Islamabad
price_list3,bath_list3,area_list3,bedroom_list3,location_list3=get_data(isd_links)

### create dataframe

In [9]:
df=pd.DataFrame({
                'price': price_list1+price_list2+price_list3,
                'Area': area_list1+area_list2+area_list3,
                'bathroom': bath_list1+bath_list2+bath_list3,
                'bedroom': bedroom_list1+bedroom_list2+bedroom_list3,
                'location': location_list1+location_list2+location_list3
                })

In [10]:
df

,price,Area,bathroom,bedroom,location
0,PKR2 Crore,4 Marla,-,-,"Eden Gardens, Faisalabad, Punjab"
1,PKR8 Crore,13 Marla,5,4,"Canal Road, Faisalabad, Punjab"
2,PKR15 Crore,1 Kanal,6,5,"Canal Road, Faisalabad, Punjab"
3,PKR5.5 Crore,14.5 Marla,5,5,"Canal Road, Faisalabad, Punjab"
4,PKR8.5 Crore,11 Marla,-,-,"Canal Road, Faisalabad, Punjab"
...,...,...,...,...,...
745,PKR12.5 Crore,1 Kanal,6,5,"DHA Defence, Islamabad, Islamabad Capital"
746,PKR11.22 Crore,1 Kanal,6,5,"DHA Defence, Islamabad, Islamabad Capital"
747,PKR28 Crore,1 Kanal,6,6,"F-8, Islamabad, Islamabad Capital"
748,PKR10.5 Crore,1 Kanal,6,5,"DHA Defence, Islamabad, Islamabad Capital"


### create csv file

In [11]:
df.to_csv('real_estate.csv', index=False)